In [1]:
%pip install spotipy
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install supabase

# MusicMatch Database Integration
from database_helper import MusicMatchDB, example_sync_user_data

# Initialize database connection
try:
    db = MusicMatchDB()
    print("✅ Database connection successful!")
    
    # Test connection
    result = db.supabase.table("users").select("count", count="exact").execute()
    print(f"📊 Current users in database: {result.count}")
    
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("Make sure you've set up Supabase and added credentials to .env file")
    db = None

Note: you may need to restart the kernel to use updated packages.
✅ Database connection successful!
✅ Database connection successful!
📊 Current users in database: 0
📊 Current users in database: 0


# Spotify Audio Features API Access

To access the Spotify Audio Features API, you need:

1. A valid access token with the proper scopes
2. Possibly a Spotify Premium account (some audio endpoints are restricted)
3. Clear token cache if authentication issues persist

The primary scopes needed are:
- `user-read-private` - Required for many analysis endpoints
- `user-library-read` - For accessing user's saved tracks

For audio analysis, you may need additional permissions or Premium status.

In [3]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os

load_dotenv()

True

In [4]:
scope = "user-library-read user-read-currently-playing app-remote-control user-modify-playback-state user-read-playback-state user-read-recently-played playlist-read-private playlist-modify-public playlist-modify-private user-follow-read user-follow-modify user-top-read user-read-playback-position user-read-email user-read-private user-read-playback-position user-read-recently-played user-library-modify user-library-read playlist-read-collaborative user-read-private user-read-email user-read-playback-state user-modify-playback-state user-read-currently-playing user-read-recently-played user-library-modify user-library-read user-follow-read user-follow-modify user-top-read playlist-read-private playlist-modify-public playlist-modify-private playlist-read-collaborative"

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    scope=scope
    ),
    requests_timeout=20
)

In [31]:
# Sync current user data to database
if db is not None:
    try:
        # Get current user and create/update in database
        user_data = sp.current_user()
        db_user = db.create_or_update_user(user_data)
        user_id = db_user["id"]
        
        print(f"✅ User synced to database!")
        print(f"📱 User: {user_data['display_name']}")
        print(f"🆔 Database ID: {user_id}")
        
        # This user_id will be used throughout the notebook
        
    except Exception as e:
        print(f"❌ User sync failed: {e}")
        user_id = None
else:
    print("⚠️ Skipping database sync - no database connection")
    user_id = None

✅ User synced to database!
📱 User: Aarush
🆔 Database ID: a4122524-5af2-4130-b8d1-c4ee3bacb326


In [32]:

# from spotipy.oauth2 import SpotifyPKCE

# # Use SpotifyPKCE instead of SpotifyOAuth for explicit PKCE flow
# auth_manager = SpotifyPKCE(
#     client_id=os.getenv("CLIENT_ID"),
#     redirect_uri=os.getenv("REDIRECT_URI"),
#     scope=scope,
#     cache_path=".spotify-token-cache",  # Where to store the token
#     open_browser=True                   # Automatically open browser for auth
# )

# # Get access token (this will trigger the auth flow if no valid token exists)
# token_info = auth_manager.get_access_token()
# # print(f"Access token retrieved (valid for {token_info['expires_in']} seconds)")

# # Create Spotify client with our auth manager
# sp_pkce = spotipy.Spotify(auth_manager=auth_manager)

# # Test the connection by getting current user profile
# try:
#     user = sp_pkce.me()
#     print(user)
#     print(f"Successfully authenticated as: {user['display_name']} ({user['id']})")
#     print(f"Account type: {user['product']}")
#     print(f"Email: {user['email']}")
#     print(f"Country: {user['country']}")
#     print(f"Followers: {user['followers']['total']}")
#     print("PKCE Authentication successful! ✅")
# except Exception as e:
#     print(f"Authentication error: {e}")



In [33]:
res = sp.current_user_saved_tracks(limit=10)
for item in res['items']:
    print(item['track']['name'])

For A Reason
Kashish
Jutti Meri (Live)
Fell For You
DESIRES
Hornn Blow
Difference
Kale Rang Da Paranda - Remix
Top Notch Gabru
Expert Jatt


In [34]:
results = sp.current_user_saved_tracks(10)
artist_ids = [item['track']['artists'][0]['id'] for item in results['items']]
# for idx, item in enumerate(results['items']):
#     track = item['track']
#     print(track['name'])
#     print(track['artists'][0]['name'], "-", track['artists'][0]['id'])
#     print()
        
print(artist_ids)

['6DARBhWbfcS9E4yJzcliqQ', '2msR4dHmBiBa99uLmuqFFk', '4E5oyNFcB3uXLkLdjYmP9Z', '5r3wPya2PpeTTsXsGhQU8O', '6LEG9Ld1aLImEFEVHdWNSB', '4ITkqBlf5eoVCOFwsJCnqo', '7GgAwYJnBBFT1WogNWf0oj', '2RoHJpBTtlOZ891LYhsRqE', '7zCChitz4Xn1O7OqXjOhhR', '1fTMfqHcXtTa0G42Wu7qH5']


In [35]:
albums = sp.current_user_saved_albums()
for idx, item in enumerate(albums['items']):
    print(item['album']['artists'][0]['name'], " – ", item['album']['name'])
    print("Popularity: ", item['album']['popularity'])
    print()

Khushi TDT  –  Victory Anthem
Popularity:  54

Various Artists  –  Crazy Love
Popularity:  4

Hassan & Roshaan  –  Day 5
Popularity:  0

The Local Train  –  Vaaqif
Popularity:  0

The Local Train  –  Aalas Ka Pedh
Popularity:  0

Zaeden  –  Genesis 1:1
Popularity:  47

The Western Ghats  –  Tu Hai
Popularity:  4

Mitraz  –  Junoon
Popularity:  48

Priyansh Srivastava  –  Sanware
Popularity:  26

Various Artists  –  T-Series Mixtape
Popularity:  41

Jassie Gill  –  Replay
Popularity:  31

Shubh  –  NO LOVE
Popularity:  0

Pritam  –  Singh is Kinng (Original Motion Picture Soundtrack)
Popularity:  44

Various Artists  –  
Popularity:  0

Tai Verdes  –  TV
Popularity:  62

Armaan Malik  –  You
Popularity:  35

The Weeknd  –  Dawn FM
Popularity:  78

Esta rola  –  Te llega al corazon
Popularity:  0

Shankar-Ehsaan-Loy  –  Rock On
Popularity:  51



In [36]:
me = sp.current_user_top_tracks(limit=15, time_range='short_term')

for idx, item in enumerate(me['items']):
    print(idx, item['artists'][0]['name'], " – ", item['name'])
    print("Popularity:", item['popularity'])
    print("Album:", item['album']['name'])
    print("Release Date:", item['album']['release_date'])
    print("External URL:", item['external_urls']['spotify'])
    print("", )
    print()


0 Ashish Bhatia  –  Kashish
Popularity: 66
Album: Kashish
Release Date: 2024-06-11
External URL: https://open.spotify.com/track/3anHs4ijBd3Iw0E0fBPwtH


1 Neha Bhasin  –  Jutti Meri (Live)
Popularity: 72
Album: Jutti Meri (Live)
Release Date: 2020-03-16
External URL: https://open.spotify.com/track/6v4OWxDDulyvjWfRXPjA8Y


2 Shubh  –  Fell For You
Popularity: 75
Album: Sicario
Release Date: 2025-01-17
External URL: https://open.spotify.com/track/5fBghXeYCGIEVuExKytoJ9


3 Karan Aujla  –  For A Reason
Popularity: 82
Album: P-POP CULTURE
Release Date: 2025-08-22
External URL: https://open.spotify.com/track/0cYohCh24y1aMjJmcS9RBl


4 Amrit Maan  –  Difference
Popularity: 53
Album: Difference
Release Date: 2018-06-08
External URL: https://open.spotify.com/track/4XO6ex8P26F5BZO5iYcpk0


5 Harrdy Sandhu  –  Hornn Blow
Popularity: 61
Album: Hornn Blow
Release Date: 2016-04-12
External URL: https://open.spotify.com/track/170bb45dHeLQdCZjAltIiB


6 AP Dhillon  –  DESIRES
Popularity: 57
Album: HI

In [37]:
artists = sp.current_user_top_artists(limit=50, time_range='long_term')
artist_ids = []
for idx, item in enumerate(artists['items']):
    print(item['name'])
    # print("Popularity:", item['popularity'])
    # print("Genres:", item['genres'])
    artist_ids.append(item['id'])
    # print()
    


Pritam
AP Dhillon
Karan Aujla
Shubh
Sidhu Moose Wala
Diljit Dosanjh
Talwiinder
King
Amit Trivedi
Zaeden
Shankar-Ehsaan-Loy
Anuv Jain
Aditya Rikhari
Satinder Sartaaj
A.R. Rahman
Badshah
Vishal-Shekhar
Lucky Ali
Aditya A
Sanam
Rochak Kohli
Salim–Sulaiman
Ayushmann Khurrana
One Direction
Kishore Kumar
Jasleen Royal
Mitraz
Yo Yo Honey Singh
Atif Aslam
Ritviz
Paradox
Dino James
Vilen
Sachin-Jigar
Guru Randhawa
Saransh Peer
DIVINE
KR$NA
Darshan Raval
Jassie Gill
Harnoor
Arijit Singh
Sachet Tandon
Lata Mangeshkar
Lash curry
Anurag Saikia
Vishal Mishra
Ed Sheeran
Jagjit Singh
Tegi Pannu


In [44]:
# Save top artists to database
if db is not None and user_id is not None:
    try:
        # Save the artists data we just retrieved
        db.save_user_top_artists(user_id, artists['items'], 'long_term')
        
        # Compute and save genre preferences
        db.compute_user_genre_preferences(user_id, 'long_term')
        
        print("✅ Top artists and genre preferences saved to database!")
        
        # Get a quick preview of saved genres
        genres_result = db.supabase.table("user_genre_preferences").select(
            "genre, weight, frequency"
        ).eq("user_id", user_id).eq("time_range", "long_term").order("weight", desc=True).limit(5).execute()
        
        print("🎵 Top 5 genres:")
        for genre in genres_result.data:
            print(f"  • {genre['genre']}: {genre['weight']:.2f} weight ({genre['frequency']} artists)")
            
    except Exception as e:
        print(f"❌ Failed to save top artists: {e}")
else:
    print("⚠️ Skipping database save - no database connection or user ID")

✅ Top artists and genre preferences saved to database!
🎵 Top 5 genres:
  • hindi pop: 0.30 weight (29 artists)
  • desi: 0.27 weight (22 artists)
  • bollywood: 0.23 weight (23 artists)
  • punjabi pop: 0.16 weight (14 artists)
  • desi hip hop: 0.15 weight (12 artists)


In [45]:
genre_freq = {}
# print("Artist IDs:", artist_ids)
for ids in artist_ids:
    artist_genre = sp.artist(ids)
    print(artist_genre['name'], '-', artist_genre['genres'], '-', ids )
    for genre in artist_genre['genres']:
        if genre in genre_freq:
            genre_freq[genre] += 1
        else:
            genre_freq[genre] = 1

print()
print("Genre Frequencies:")
for genre, freq in genre_freq.items():
    print(f"{genre}: {freq}")

Pritam - ['bollywood', 'hindi pop', 'desi'] - 1wRPtKGflJrBx9BmLsSwlU
AP Dhillon - ['punjabi hip hop', 'punjabi pop', 'desi', 'bhangra', 'desi hip hop'] - 6LEG9Ld1aLImEFEVHdWNSB
Karan Aujla - ['punjabi pop', 'punjabi hip hop', 'bhangra', 'desi hip hop', 'desi'] - 6DARBhWbfcS9E4yJzcliqQ
Shubh - ['punjabi hip hop', 'punjabi pop', 'desi hip hop', 'desi', 'hindi hip hop'] - 5r3wPya2PpeTTsXsGhQU8O
Karan Aujla - ['punjabi pop', 'punjabi hip hop', 'bhangra', 'desi hip hop', 'desi'] - 6DARBhWbfcS9E4yJzcliqQ
Shubh - ['punjabi hip hop', 'punjabi pop', 'desi hip hop', 'desi', 'hindi hip hop'] - 5r3wPya2PpeTTsXsGhQU8O
Sidhu Moose Wala - ['punjabi hip hop', 'punjabi pop', 'bhangra', 'desi hip hop'] - 4PULA4EFzYTrxYvOVlwpiQ
Diljit Dosanjh - ['bhangra', 'punjabi pop', 'punjabi hip hop', 'desi', 'bollywood'] - 2FKWNmZWDBZR4dE5KX4plR
Sidhu Moose Wala - ['punjabi hip hop', 'punjabi pop', 'bhangra', 'desi hip hop'] - 4PULA4EFzYTrxYvOVlwpiQ
Diljit Dosanjh - ['bhangra', 'punjabi pop', 'punjabi hip hop', 'de

In [46]:
import time

genre_freq = {}
print("Artist IDs:", artist_ids)

# Process in smaller batches with delays between requests
batch_size = 5  # Process 5 artists at a time
for i in range(0, len(artist_ids), batch_size):
    batch = artist_ids[i:i+batch_size]
    
    for ids in batch:
        try:
            artist_genre = sp.artist(ids)
            for genre in artist_genre['genres']:
                if genre in genre_freq:
                    genre_freq[genre] += 1
                else:
                    genre_freq[genre] = 1
            # Short delay between requests
            time.sleep(0.5)
        except Exception as e:
            print(f"Error processing artist ID {ids}: {e}")
    
    # Longer delay between batches
    print(f"Processed batch {i//batch_size + 1}/{(len(artist_ids) + batch_size - 1)//batch_size}")
    time.sleep(2)

print()
print("Genre Frequencies:")
for genre, freq in sorted(genre_freq.items(), key=lambda x: x[1], reverse=True):
    print(f"{genre}: {freq}")

Artist IDs: ['1wRPtKGflJrBx9BmLsSwlU', '6LEG9Ld1aLImEFEVHdWNSB', '6DARBhWbfcS9E4yJzcliqQ', '5r3wPya2PpeTTsXsGhQU8O', '4PULA4EFzYTrxYvOVlwpiQ', '2FKWNmZWDBZR4dE5KX4plR', '6QoCrBHsojKnOrsGNfRcTN', '5NHm4TU5Twz7owibYxJfFU', '7HCqGPJcQTyGJ2yqntbuyr', '5lMNphVhMLvhFmTWiKiLA2', '0L5GV6LN8SWWUWIdBbTLTZ', '4gdMJYnopf2nEUcanAwstx', '3ozYqVCLohfpXIhalkhM8D', '4rgw8A5vcYinpZLDKHrEdV', '1mYsTxnqsietFxj1OgoGbG', '0y59o4v8uw5crbN9M3JiL1', '6Mv8GjQa7LKUGCAqa9qqdb', '2L16nDKTxhFGaDriR2AHTB', '4wwYGgSpeBtvk5WX6HBqzw', '7o7doCwqft91WC690aglWC', '3dN9MQpjIyNxyeRfz4EDZe', '6ohaQzKaXrobAL8paLSaxq', '7qHsapL39aTQsPhixtzVvy', '4AK6F7OLvEQ5QYCBNiQWHq', '0GF4shudTAFv8ak9eWdd4Y', '74OaRjmyh0XyRZsQQQ5l7c', '3iGhlvzpXc0UHBQ7klAItX', '7uIbLdzzSEqnX0Pkrb56cR', '2oSONSC9zQ4UonDKnLqksx', '72beYOeW2sb2yfcS4JsRvb', '3fWcIRZlzhMl2YNACMvHui', '45PG2L6Fh2XvYL4ONzpdoW', '5gVozagAcRKYCeAVnlC3Nk', '1mBydYMVBECdDmMfE2sEUO', '5rQoBDKFnd1n6BkdbgVaRL', '5lpH8m90JdnCrHce4janvf', '4Ai0pGz6GhQavjzaRhPTvz', '5C1S9XwxMuuCciutwMhp5t',

In [47]:

playlists = sp.current_user_playlists()
print("Playlists:")
for playlist in playlists['items']:
    print(playlist['name'], '--', playlist['id'])

Playlists:
Car -- 6PeJhlIkXf4bq4gpUQJFVl
PUNJABI GYM HITS 🔥 | WORKOUT PUMP SONGS | NEW PUNJABI ATTITUDE SONGS 2025  -- 1LIowjORrNqFFyXYqK0JvE
🌌 -- 2QXI7vr0ypYxhJRhPiooSg
elite songs -- 5hENS6QaEjmgfSt4zjlDul
Second South Carolina String Band  -- 7mi82SdG8V0X94GUPHveFf
Indie 'n' Chill -- 2cdkBS6JAmaKETJgS6npOU
X -- 3cMY7YaIVPNZk5WNAUzrkM
Hindi motivation songs❤️‍🔥 -- 142zS6xmGAu11tVBfSlWR2
Gedi Da Tashan -- 1OF0FbvIXKawGBcWUQnYms
DIWALI PUJA -- 05v5XNbC9QvknSYUKBBL80
aakhon ko teri aadat hai :) -- 5JTTcFtWXxy9QfNcsOqO0v
my coke studio favourites -- 0OzGkp1IUCaSb2Tl0VOU1V
Where'd All the Time Go? -- 494l0zKf2mbUUNxj11iQEH
WOMEN'S ERA -- 1AC1qLroAxDMbDlLs97jqc
Viva La Vida -- 4wCATZ4WS5jJG1RsArtBev
Indie -- 2dv8RPPR7weaLXspn86gae
 -- 1T0GMrIfexr90ZmFcihsqf
3 AM -- 407Neky5CtG3DVObeEfZyl
2000s hits✨ -- 4fIIQF84uBpiBqjE1T7eQv
feel good -- 50GA5ij1bly1pF8LaHdhNJ
Punjabi Popular hit songs 2025 🔥 Latest & Trending -- 3lot3HM1GHmXDzJFt1cjTb
Dear Diary, We fell apart -- 5IoBsACYpvzQInfIatGkbR
Fa

In [48]:
playlist_tracks = sp.playlist_tracks('3YTRLL4fA0n824u6jBCWjy')

for idx, item in enumerate(playlist_tracks['items']):
    track = item['track']
    added = item['added_at']
    # convert added to a readable format
    added_date = time.strftime('%Y-%m-%d %H:%M:%S', time.strptime(added, '%Y-%m-%dT%H:%M:%SZ'))
    
    print(idx+1, track['name'], " – ", track['artists'][0]['name'])
    print("Popularity:", track['popularity'])
    print("Album:", track['album']['name'])
    print("Date added:", added_date)
    print("Release Date:", track['album']['release_date'])
    print("External URL:", track['external_urls']['spotify'])
    print("", )
    print()

1 Starlight (Taylor's Version)  –  Taylor Swift
Popularity: 62
Album: Red (Taylor's Version)
Date added: 2021-11-12 05:26:12
Release Date: 2021-11-12
External URL: https://open.spotify.com/track/7A2cNLRT0YJc1yjxHlKihs


2 Babe (Taylor's Version) (From The Vault)  –  Taylor Swift
Popularity: 66
Album: Red (Taylor's Version)
Date added: 2021-11-12 05:13:33
Release Date: 2021-11-12
External URL: https://open.spotify.com/track/0v4z1tuZvn6LGknom9Qx7d


3 First Times  –  Ed Sheeran
Popularity: 61
Album: =
Date added: 2021-10-29 17:47:52
Release Date: 2021-10-25
External URL: https://open.spotify.com/track/5QYnNhTKsN3kE7OaqILA1U


4 Collide  –  Ed Sheeran
Popularity: 56
Album: =
Date added: 2021-10-29 17:47:47
Release Date: 2021-10-25
External URL: https://open.spotify.com/track/5sv0WnUs74Orn6GoPmC5im


5 Call It What You Want  –  Taylor Swift
Popularity: 77
Album: reputation
Date added: 2021-10-03 07:09:18
Release Date: 2017-11-10
External URL: https://open.spotify.com/track/1GwMQaZz6Au3QLDb

In [49]:
# Test: View your music profile from database
if db is not None and user_id is not None:
    try:
        profile = db.get_user_music_profile(user_id)
        
        print("🎵 YOUR MUSIC PROFILE:")
        print(f"📊 Saved tracks: {profile['saved_tracks_count']}")
        print(f"🎭 Top genres: {len(profile['genre_preferences'])}")
        print(f"👨‍🎤 Top artists: {len(profile['top_artists'])}")
        
        print("\n🔥 Your top genres:")
        for genre in profile['genre_preferences'][:5]:
            print(f"  • {genre['genre']}: {genre['weight']:.2f}")
            
        print("\n⭐ Your top artists:")
        for artist in profile['top_artists'][:5]:
            print(f"  {artist['position']}. {artist['artists']['name']}")
            
    except Exception as e:
        print(f"❌ Failed to get profile: {e}")
else:
    print("⚠️ No database connection to test")

🎵 YOUR MUSIC PROFILE:
📊 Saved tracks: 0
🎭 Top genres: 24
👨‍🎤 Top artists: 50

🔥 Your top genres:
  • hindi pop: 0.30
  • desi: 0.27
  • bollywood: 0.23
  • punjabi pop: 0.16
  • hindi indie: 0.15

⭐ Your top artists:
  1. Pritam
  2. AP Dhillon
  3. Karan Aujla
  4. Shubh
  5. Sidhu Moose Wala
